# Pipeline V8 — Fase 3: Rotulação/Re-rotulação Databricks (definitivo)

**Spec**: `specs/004-melhoria-geracao-dados-cnn/` (T-A3-007). **Execução: Databricks.**

Calcula `melhor_jogada`, `score_melhor_jogada` e `depth_melhor_jogada` para qualquer
lote de NPZs com schema v2-a3 (12 canais + 3 escalares de cadeias já presentes).

**Uso primário**: rotular NPZs recém-gerados pelo pipeline V7 DAC (Fase 1) após
enriquecimento pela Fase 2. Também serve para re-rotular datasets existentes com
profundidade maior.

### Estratégia de profundidade adaptativa

```
depth_estado = DEPTH_ADAPTATIVO  se  arestas_livres > 11  E  prof_min > 11
             = DEPTH_PADRAO       caso contrário

onde:  arestas_livres = 31 − qtd_tracos
       prof_min       = total_caixas_cadeias_longas + 2 × qtd_cadeias_longas
```

O Minimax para naturalmente quando o jogo termina; `DEPTH_ADAPTATIVO=20` resolve
qualquer estado com ≤ 19 arestas livres (máximo observado no dataset).

### Otimizações (baseadas em Geracao_Amostras_v7_adaptativo_Fase_2_HighPerf.ipynb)

1. **Databricks Serverless**: usa `mapInPandas` — sem RDDs, evita `JVM_ATTRIBUTE_NOT_SUPPORTED`.
2. **Profundidade por estado**: coluna `depth` enviada a cada worker — estados com cadeias longas
   recebem `DEPTH_ADAPTATIVO`, demais recebem `DEPTH_PADRAO`.
3. **Checkpointing (lotes de `LOTE_ARQUIVOS`)**: em caso de queda do cluster, retoma do último lote.
4. **Gravação atômica**: `.tmp.npz` + `os.replace()` — nunca corrompe originais.
5. **Preservação total do schema v2-a3**: canais, nomes_canais e 3 escalares são mantidos.

> **⚠️ ALERTA DE MANUTENÇÃO: BUGS ALGÓRITMICOS NO BITBOARD (Maio/2026)**
>
> O Minimax via Bitboard neste notebook possui dois bugs que foram corrigidos e
> **NÃO DEVEM ser revertidos ou esquecidos** ao adaptar este código:
>
> **Bug 1 — Caixas Pré-Fechadas (Falsos Positivos):** ao contar caixas completadas
> por uma jogada, é OBRIGATÓRIO verificar que a caixa *não estava fechada antes*:
> ```python
> # CORRETO:
> closed = sum(1 for bm in edge_boxes[i] if new_e & bm == bm and edges & bm != bm)
> # ERRADO (conta caixas já fechadas antes da jogada):
> closed = sum(1 for bm in edge_boxes[i] if new_e & bm == bm)
> ```
>
> **Bug 2 — Offsets na Poda Alpha-Beta Incremental:** este Bitboard retorna scores
> *incrementais* (não absolutos). Ao chamar a subárvore após capturar `closed` caixas,
> os limites alpha/beta DEVEM ser ajustados com offset de `−closed`:
> ```python
> # CORRETO (maximizando, capturou `closed` caixas — mesma vez):
> val = closed + solve_minimax_bitboard(new_e, depth-1, alpha - closed, beta - closed, True, tt)
> # ERRADO (repassa alpha/beta puro — causa poda prematura e escolhe jogadas perdedoras):
> val = closed + solve_minimax_bitboard(new_e, depth-1, alpha, beta, True, tt)
> ```

In [0]:
import os
import time
import random
import numpy as np
import pandas as pd
from pathlib import Path
from pyspark.sql import functions as F

# ── Configurações ──────────────────────────────────────────────────────────────
# Path do diretório NPZ no DBFS ou montagem S3/Azure.
INPUT_DIR = Path('/Workspace/Users/c092820@corp.caixa.gov.br/CNN/profundidade_minimax_11_v7_adaptativo')

# Profundidade padrão para estados sem cadeias longas problemáticas.
DEPTH_PADRAO = 11

# Profundidade usada quando arestas_livres > 11 E prof_min > 11.
# Minimax para naturalmente — p=20 resolve qualquer estado com <= 19 arestas livres.
DEPTH_ADAPTATIVO = 20

# Número de NPZs por lote (checkpoint). ~4-8 NPZs = ~20-40k estados por lote.
LOTE_ARQUIVOS = 16
NUM_PARTICOES = 360

# Se True: re-rotula TODOS os estados, inclusive os já rotulados.
# Se False: pula estados onde depth_melhor_jogada[i] >= depth_necessaria[i].
FORCAR_REGRAVAR = False

print(f'INPUT_DIR        = {INPUT_DIR}')
print(f'DEPTH_PADRAO     = {DEPTH_PADRAO}')
print(f'DEPTH_ADAPTATIVO = {DEPTH_ADAPTATIVO}')
print(f'LOTE_ARQUIVOS    = {LOTE_ARQUIVOS}')
print(f'FORCAR_REGRAVAR  = {FORCAR_REGRAVAR}')

INPUT_DIR        = /Workspace/Users/c092820@corp.caixa.gov.br/CNN/profundidade_minimax_11_v7_adaptativo
DEPTH_PADRAO     = 11
DEPTH_ADAPTATIVO = 20
LOTE_ARQUIVOS    = 16
FORCAR_REGRAVAR  = False


In [ ]:
# ── Diagnóstico + pré-carga do cache global ───────────────────────────────────
# Varredura ÚNICA em todos os NPZs:
#   1. Conta estados totais / pendentes (bruto e distintos).
#   2. Constrói cache_global com resultados já calculados em NPZs rotulados.
#      Tabuleiros idênticos em NPZs novos recebem dados do cache sem Minimax.
#   3. Monta a lista 'pendentes' de NPZs com ao menos um estado por calcular.
#
# Memória: cache_global ocupa ~200-250 MB para 760 k estados rotulados.
# pending_hashes usa hash de 64 bits — colisão improvável (~0,06% p/ 2 M itens).

SUFIXOS_SIMETRIA = ('_refH', '_refV', '_r180')
todos_npzs = sorted(INPUT_DIR.glob('dataset_pequeno_*.npz'))
npzs_originais = [
    p for p in todos_npzs
    if not any(p.stem.endswith(s) for s in SUFIXOS_SIMETRIA)
    and '.tmp.' not in p.name
]

print(f'NPZs originais: {len(npzs_originais)}')

# cache_global: (estado_bytes, depth_necessaria) -> (melhor_jogada, scores_array)
cache_global: dict = {}

# Lista de NPZs com ao menos um estado pendente.
pendentes: list = []

# Hashes dos pares (estado_bytes, depth_necessaria) pendentes para contar
# tabuleiros distintos sem guardar os bytes completos em memória.
pending_hashes: set = set()

n_padrao = 0
n_adaptativo = 0
n_pendentes_padrao = 0
n_pendentes_adaptativo = 0

for path in npzs_originais:
    with np.load(path, allow_pickle=False) as d:
        estados_arr = d['estados']
        qtd_tracos  = d['qtd_tracos'].astype(np.int32)
        qtd_cad     = d['qtd_cadeias_longas'].astype(np.int32)
        total_cad   = d['total_caixas_cadeias_longas'].astype(np.int32)
        depth_atual = d['depth_melhor_jogada'].astype(np.int32) if 'depth_melhor_jogada' in d.files else np.zeros(len(qtd_tracos), dtype=np.int32)
        rotulados   = (d['melhor_jogada'] != '') if 'melhor_jogada' in d.files else np.zeros(len(qtd_tracos), dtype=bool)
        mj_raw  = d['melhor_jogada']       if 'melhor_jogada'       in d.files else None
        smj_raw = d['score_melhor_jogada'] if 'score_melhor_jogada' in d.files else None

    arestas_livres = 31 - qtd_tracos
    prof_min = total_cad + 2 * qtd_cad
    precisa_adaptativo = (arestas_livres > 11) & (prof_min > 11)
    depth_necessaria = np.where(precisa_adaptativo, DEPTH_ADAPTATIVO, DEPTH_PADRAO)

    n_padrao      += int((~precisa_adaptativo).sum())
    n_adaptativo  += int(precisa_adaptativo.sum())

    if FORCAR_REGRAVAR:
        pendente = np.ones(len(qtd_tracos), dtype=bool)
    else:
        pendente = ~rotulados | (depth_atual < depth_necessaria)

    n_pendentes_padrao      += int((pendente & ~precisa_adaptativo).sum())
    n_pendentes_adaptativo  += int((pendente & precisa_adaptativo).sum())

    # Alimenta cache_global com resultados já calculados e com depth suficiente.
    if not FORCAR_REGRAVAR and mj_raw is not None and smj_raw is not None:
        for j in range(len(estados_arr)):
            if rotulados[j] and depth_atual[j] >= depth_necessaria[j]:
                chave = (estados_arr[j].tobytes(), int(depth_necessaria[j]))
                if chave not in cache_global:
                    cache_global[chave] = (str(mj_raw[j]), smj_raw[j].copy())

    # Conta tabuleiros distintos pendentes via hash de 64 bits.
    for j in range(len(estados_arr)):
        if pendente[j]:
            h = hash(estados_arr[j].tobytes()) ^ (int(depth_necessaria[j]) * 0x9E3779B97F4A7C15)
            pending_hashes.add(h)

    if bool(pendente.any()):
        pendentes.append(path)

total_estados   = n_padrao + n_adaptativo
total_pendentes = n_pendentes_padrao + n_pendentes_adaptativo
pct = 100 * total_pendentes / max(total_estados, 1)

print(f'\nTotal de estados       : {total_estados:,}')
print(f'  Depth {DEPTH_PADRAO} (padrão)   : {n_padrao:,}')
print(f'  Depth {DEPTH_ADAPTATIVO} (adaptativo) : {n_adaptativo:,}')
print(f'\nPendentes (bruto)      : {total_pendentes:,} ({pct:.2f}%)')
print(f'  com depth {DEPTH_PADRAO}         : {n_pendentes_padrao:,}')
print(f'  com depth {DEPTH_ADAPTATIVO}         : {n_pendentes_adaptativo:,}')
print(f'\nTabuleiros DISTINTOS pendentes : {len(pending_hashes):,}')
print(f'Cache global pré-carregado     : {len(cache_global):,} resultados')
print(f'\nArquivos já completos : {len(npzs_originais) - len(pendentes)}')
print(f'Arquivos na fila      : {len(pendentes)}')

In [0]:
# ── Worker PySpark: Minimax Bitboard com profundidade por estado ───────────────
# Toda a lógica está inline para evitar erros de serialização (Pickle) no
# Databricks Serverless. Ambos os bug fixes do Bitboard estão aplicados.

def processar_lote_pandas(iterator):
    """
    Função enviada aos Workers PySpark via mapInPandas.
    Recebe DataFrame com colunas: estado_bytes (BINARY), depth (INT).
    Retorna: estado_bytes, depth, melhor_jogada (STRING), scores (ARRAY<FLOAT>).
    """
    import random
    import numpy as np
    import pandas as pd

    # ── Reconstrói tabelas de Bitboard localmente no Worker ──────────────────
    h, w = 9, 7
    edge_to_bit = {}
    bit_to_label = {}
    bit_idx = 0
    for r in range(h):
        for c in range(w):
            if (r % 2 == 0 and c % 2 == 1) or (r % 2 == 1 and c % 2 == 0):
                edge_to_bit[(r, c)] = bit_idx
                bit_to_label[bit_idx] = f'H_{r}_{c}' if r % 2 == 0 else f'V_{r}_{c}'
                bit_idx += 1

    n_edges = bit_idx
    all_mask = (1 << n_edges) - 1

    box_masks = []
    for r in range(1, h, 2):
        for c in range(1, w, 2):
            mask = (1 << edge_to_bit[(r-1, c)]) | (1 << edge_to_bit[(r+1, c)]) | \
                   (1 << edge_to_bit[(r, c-1)]) | (1 << edge_to_bit[(r, c+1)])
            box_masks.append(mask)

    edge_boxes = [tuple(bm for bm in box_masks if bm & (1 << b)) for b in range(n_edges)]

    # ── Minimax Bitboard ──────────────────────────────────────────────────────
    # Fix 1: `edges & bm != bm` — exclui caixas pré-fechadas.
    # Fix 2: offset alpha/beta por `closed` — poda incremental correta.
    def solve_minimax_bitboard(edges, depth, alpha, beta, maximizing, tt):
        if depth == 0 or edges == all_mask:
            return 0

        tt_key = (edges, depth, maximizing)
        if tt_key in tt:
            flag, val = tt[tt_key]
            if flag == 0: return val
            if flag == 1 and val >= beta: return val
            if flag == 2 and val <= alpha: return val

        moves = []
        for i in range(n_edges):
            if not (edges & (1 << i)):
                ne = edges | (1 << i)
                # Fix 1: conta apenas caixas que foram FECHADAS AGORA
                closed = sum(1 for bm in edge_boxes[i] if ne & bm == bm and edges & bm != bm)
                moves.append((i, closed))
        moves.sort(key=lambda x: x[1], reverse=True)

        orig_alpha = alpha
        orig_beta = beta
        best_val = -10000 if maximizing else 10000

        for bit, closed in moves:
            new_e = edges | (1 << bit)
            if maximizing:
                if closed > 0:
                    # Fix 2: offset alpha/beta incremental — mesma vez (capturou e joga de novo)
                    val = closed + solve_minimax_bitboard(new_e, depth-1, alpha - closed, beta - closed, True, tt)
                else:
                    val = solve_minimax_bitboard(new_e, depth-1, alpha, beta, False, tt)
                best_val = max(best_val, val)
                alpha = max(alpha, best_val)
            else:
                if closed > 0:
                    # Fix 2: adversário capturou — offset invertido
                    val = -closed + solve_minimax_bitboard(new_e, depth-1, alpha + closed, beta + closed, False, tt)
                else:
                    val = solve_minimax_bitboard(new_e, depth-1, alpha, beta, True, tt)
                best_val = min(best_val, val)
                beta = min(beta, best_val)
            if beta <= alpha:
                break

        # TT com flags EXACT / LOWERBOUND / UPPERBOUND
        if maximizing:
            flag = 2 if best_val <= orig_alpha else (1 if best_val >= beta else 0)
        else:
            flag = 1 if best_val >= orig_beta else (2 if best_val <= alpha else 0)
        tt[tt_key] = (flag, best_val)
        return best_val

    # ── Processamento dos DataFrames recebidos do Spark ───────────────────────
    for pdf in iterator:
        resultados = []
        for row in pdf.itertuples(index=False):
            estado_bytes = bytes(row.estado_bytes)
            depth = int(row.depth)

            mat = np.frombuffer(estado_bytes, dtype=np.int8).reshape(9, 7)
            edges = 0
            idx = 0
            for r in range(9):
                for c in range(7):
                    if (r % 2 == 0 and c % 2 == 1) or (r % 2 == 1 and c % 2 == 0):
                        if mat[r, c] == 9:
                            edges |= (1 << idx)
                        idx += 1

            # TT reiniciada por estado (não compartilhar entre tabuleiros diferentes)
            tt = {}
            scores = np.full(31, -1_000_000_000.0, dtype=np.float32)

            for i in range(31):
                if not (edges & (1 << i)):
                    new_e = edges | (1 << i)
                    closed = sum(1 for bm in edge_boxes[i] if new_e & bm == bm and edges & bm != bm)
                    if closed > 0:
                        res = closed + solve_minimax_bitboard(new_e, depth-1, -10001, 10001, True, tt)
                    else:
                        res = solve_minimax_bitboard(new_e, depth-1, -10001, 10001, False, tt)
                    scores[i] = float(res)

            validos = [i for i, s in enumerate(scores) if s > -1e8]
            if not validos:
                melhor_rotulo = ''
            else:
                max_s = max(scores[i] for i in validos)
                best_idx = random.choice([i for i in validos if scores[i] == max_s])
                melhor_rotulo = bit_to_label[best_idx]

            resultados.append({
                'estado_bytes': estado_bytes,
                'depth': depth,
                'melhor_jogada': melhor_rotulo,
                'scores': scores.tolist(),
            })
        yield pd.DataFrame(resultados)

print('Worker processar_lote_pandas() definido.')

Worker processar_lote_pandas() definido.


In [ ]:
# ── Fila de NPZs pendentes ─────────────────────────────────────────────────────
# A fila 'pendentes' e o 'cache_global' foram construídos na célula de diagnóstico.
# Execute esta célula para confirmar o estado antes de iniciar o loop.

print(f'Arquivos já completos : {len(npzs_originais) - len(pendentes)}')
print(f'Arquivos na fila      : {len(pendentes)}')
print(f'Cache global          : {len(cache_global):,} resultados pré-carregados')

In [ ]:
# ── Loop de processamento em lotes ────────────────────────────────────────────

spark.conf.set('spark.databricks.execution.timeout', 7200)

schema_spark = 'estado_bytes BINARY, depth INT, melhor_jogada STRING, scores ARRAY<FLOAT>'

for i in range(0, len(pendentes), LOTE_ARQUIVOS):
    lote_paths = pendentes[i : i + LOTE_ARQUIVOS]

    # Coleta pares únicos (estado_bytes, depth_necessaria) pendentes neste lote.
    estados_para_calcular = {}  # (bytes, depth) -> depth  (deduplicação intra-lote)

    for path in lote_paths:
        with np.load(path, allow_pickle=False) as d:
            qtd_tracos  = d['qtd_tracos'].astype(np.int32)
            qtd_cad     = d['qtd_cadeias_longas'].astype(np.int32)
            total_cad   = d['total_caixas_cadeias_longas'].astype(np.int32)
            depth_atual = d['depth_melhor_jogada'].astype(np.int32) if 'depth_melhor_jogada' in d.files else np.zeros(len(qtd_tracos), dtype=np.int32)
            rotulados   = d['melhor_jogada'] != '' if 'melhor_jogada' in d.files else np.zeros(len(qtd_tracos), dtype=bool)
            estados_arr = d['estados']

        arestas_livres = 31 - qtd_tracos
        prof_min = total_cad + 2 * qtd_cad
        precisa_adaptativo = (arestas_livres > 11) & (prof_min > 11)
        depth_necessaria = np.where(precisa_adaptativo, DEPTH_ADAPTATIVO, DEPTH_PADRAO)

        for j, estado in enumerate(estados_arr):
            if not FORCAR_REGRAVAR and rotulados[j] and depth_atual[j] >= depth_necessaria[j]:
                continue
            chave = (estado.tobytes(), int(depth_necessaria[j]))
            estados_para_calcular[chave] = int(depth_necessaria[j])

    print(f'--- Lote {i // LOTE_ARQUIVOS + 1}/{(len(pendentes) + LOTE_ARQUIVOS - 1) // LOTE_ARQUIVOS} ({len(lote_paths)} NPZs) ---')
    print(f'    Estados únicos a calcular: {len(estados_para_calcular)}')

    if not estados_para_calcular:
        print('    Nenhum pendente neste lote. Pulando.')
        continue

    # ── Separa: servidos do cache_global vs precisam do Spark ─────────────────
    args_cache: list = []
    args_compute: list = []
    for chave in estados_para_calcular.keys():
        if not FORCAR_REGRAVAR and chave in cache_global:
            args_cache.append(chave)
        else:
            args_compute.append(chave)

    print(f'    Cache global : {len(args_cache):,} | A computar via Spark: {len(args_compute):,}')

    # Resultados servidos do cache (sem custo de Spark).
    resultados_lote: list = [
        (bytes(chave[0]), chave[1], cache_global[chave][0], cache_global[chave][1].tolist())
        for chave in args_cache
    ]

    # Computa o restante via Spark.
    if args_compute:
        t_lote = time.time()
        pdf_in = pd.DataFrame([
            {'estado_bytes': estado_b, 'depth': depth_val}
            for estado_b, depth_val in args_compute
        ])
        df_spark = spark.createDataFrame(pdf_in)
        # Embaralhamos para que cada partição receba estados de depth variado (p=11 e p=20).
        df_spark = df_spark.orderBy(F.rand()).repartition(NUM_PARTICOES)
        print(f'    Partições: {NUM_PARTICOES} (~{len(args_compute) // NUM_PARTICOES} estados/partição)')

        df_result = df_spark.mapInPandas(processar_lote_pandas, schema=schema_spark)
        resultados_computados = df_result.collect()

        # Adiciona ao cache_global para beneficiar lotes futuros.
        for row in resultados_computados:
            chave = (bytes(row.estado_bytes), int(row.depth))
            if chave not in cache_global:
                cache_global[chave] = (row.melhor_jogada, np.array(row.scores, dtype=np.float32))

        resultados_lote.extend([
            (bytes(row.estado_bytes), row.depth, row.melhor_jogada, list(row.scores))
            for row in resultados_computados
        ])
        print(f'    Spark: {len(resultados_computados):,} resultados em {time.time() - t_lote:.1f}s.')
    else:
        print('    Todos servidos do cache_global. Gravando...')

    # Cache local do lote: (estado_bytes, depth) -> (melhor_jogada, scores_array)
    cache = {
        (eb, int(dep)): (mj, np.array(sc, dtype=np.float32))
        for eb, dep, mj, sc in resultados_lote
    }

    print(f'    Total no lote: {len(cache):,} resultados. Gravando...')

    # Gravação atômica: preserva TODO o schema v2-a3.
    for path in lote_paths:
        with np.load(path, allow_pickle=False) as d:
            dados = dict(d)

        estados_arr = dados['estados']
        N = len(estados_arr)
        qtd_tracos  = dados['qtd_tracos'].astype(np.int32)
        qtd_cad     = dados['qtd_cadeias_longas'].astype(np.int32)
        total_cad   = dados['total_caixas_cadeias_longas'].astype(np.int32)

        arestas_livres = 31 - qtd_tracos
        prof_min = total_cad + 2 * qtd_cad
        precisa_adaptativo = (arestas_livres > 11) & (prof_min > 11)
        depth_necessaria = np.where(precisa_adaptativo, DEPTH_ADAPTATIVO, DEPTH_PADRAO)

        depth_atual = dados['depth_melhor_jogada'].astype(np.int32) if 'depth_melhor_jogada' in dados else np.zeros(N, dtype=np.int32)
        rotulados   = dados['melhor_jogada'] != '' if 'melhor_jogada' in dados else np.zeros(N, dtype=bool)

        mj_arr  = np.array(list(dados.get('melhor_jogada', [''] * N)), dtype='<U5')
        smj_arr = dados.get('score_melhor_jogada', np.full((N, 31), -1e9, dtype=np.float32)).copy()
        dm_arr  = depth_atual.copy().astype(np.int8)

        for j, estado in enumerate(estados_arr):
            if not FORCAR_REGRAVAR and rotulados[j] and depth_atual[j] >= depth_necessaria[j]:
                continue
            chave = (estado.tobytes(), int(depth_necessaria[j]))
            if chave in cache:
                mj_arr[j], smj_arr[j] = cache[chave]
                dm_arr[j] = np.int8(depth_necessaria[j])

        dados['melhor_jogada']        = mj_arr
        dados['score_melhor_jogada']  = smj_arr
        dados['depth_melhor_jogada']  = dm_arr

        tmp_path = path.with_name(path.stem + '.tmp.npz')
        np.savez_compressed(tmp_path, **dados)
        os.replace(tmp_path, path)
        print(f'    -> {path.name} gravado.')

print('\nTodo o processamento finalizado!')

In [0]:
# ── Auditoria final ───────────────────────────────────────────────────────────

erros = []
total_padrao = 0
total_adaptativo = 0

for path in npzs_originais:
    with np.load(path, allow_pickle=False) as d:
        if 'melhor_jogada' not in d.files or (d['melhor_jogada'] == '').any():
            erros.append(f'{path.name}: melhor_jogada vazia')
            continue
        dm = d['depth_melhor_jogada'].astype(np.int32)
        qtd_tracos = d['qtd_tracos'].astype(np.int32)
        qtd_cad    = d['qtd_cadeias_longas'].astype(np.int32)
        total_cad  = d['total_caixas_cadeias_longas'].astype(np.int32)

    arestas_livres = 31 - qtd_tracos
    prof_min = total_cad + 2 * qtd_cad
    precisa_adaptativo = (arestas_livres > 11) & (prof_min > 11)
    depth_necessaria = np.where(precisa_adaptativo, DEPTH_ADAPTATIVO, DEPTH_PADRAO)

    insuficiente = dm < depth_necessaria
    if insuficiente.any():
        erros.append(f'{path.name}: {insuficiente.sum()} estados com depth < necessária')

    total_padrao      += int((dm == DEPTH_PADRAO).sum())
    total_adaptativo  += int((dm == DEPTH_ADAPTATIVO).sum())

if erros:
    print(f'FALHA: {len(erros)} problema(s):')
    for e in erros[:20]:
        print(f'  {e}')
else:
    print(f'OK: {len(npzs_originais)} NPZs auditados.')
    print(f'  Estados com depth {DEPTH_PADRAO} (padrão)   : {total_padrao:,}')
    print(f'  Estados com depth {DEPTH_ADAPTATIVO} (adaptativo) : {total_adaptativo:,}')
    print('Pronto para fase4_augmentacao_simetria.ipynb.')

OK: 152 NPZs auditados.
  Estados com depth 11 (padrão)   : 747,098
  Estados com depth 20 (adaptativo) : 11,542
Pronto para fase4_augmentacao_simetria.ipynb.
